In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_practitioners;
SET search_path TO qdp_practitioners;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp_practitioners',
  '0.1',
  'practitioners.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_practitioner table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_practitioner CASCADE;

CREATE TABLE hub_practitioner (
    practitioner_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    practitioner_entity_id VARCHAR,
    person_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_practitioner ADD CONSTRAINT pk_practitioner_hubpractitioner_practitionerid PRIMARY KEY (practitioner_id);

COMMENT ON COLUMN hub_practitioner.practitioner_id IS 'Practitioner Identity';
COMMENT ON COLUMN hub_practitioner.practitioner_entity_id IS 'The unique Practitioner Composite Key identifier, generated during Quantexa Processing. May comprise multiple source identifiers for merged source practitioner records';
COMMENT ON COLUMN hub_practitioner.person_entity_id IS 'The unique Practitioner as a Person Composite Key identifier, generated during Quantexa Processing. May comprise multiple source identifiers for merged source practitioner records';




-- -----------------------------------------------------------
-- create sat_practitioner table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_practitioner;

CREATE TABLE sat_practitioner (
    sat_practitioner_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    practitioner_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	-- Cross-schema reference to qdp_persons.hub_person.person_id
	person_id BIGINT,
	active BOOLEAN DEFAULT TRUE,
	photo BYTEA,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_practitioner ADD CONSTRAINT pk_practitioner_satpractitioner_satpractitionerid PRIMARY KEY (sat_practitioner_id);
ALTER TABLE sat_practitioner ADD CONSTRAINT fk_practitioner_satpractitioner_hubpractitioner_practitionerid FOREIGN KEY (practitioner_id) REFERENCES hub_practitioner(practitioner_id);

COMMENT ON COLUMN sat_practitioner.practitioner_id IS 'Practitioner Identity';
COMMENT ON COLUMN sat_practitioner.sat_practitioner_id IS 'Practitioner Identity Sat';
COMMENT ON COLUMN sat_practitioner.person_id IS 'Cross-schema reference to qdp_persons.hub_person.[ersonindividual]_id';
COMMENT ON COLUMN sat_practitioner.active IS 'Whether this practitioner''s record is in active use. See https://build.fhir.org/practitioner-definitions.html#Practitioner.active';
COMMENT ON COLUMN sat_practitioner.photo IS 'Image of the person. See https://build.fhir.org/practitioner-definitions.html#Practitioner.photo';
COMMENT ON COLUMN sat_practitioner.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';




-- -----------------------------------------------------------
-- create sat_practitioner_qualification table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_practitioner_qualification;

CREATE TABLE sat_practitioner_qualification (
	practitioner_qualification_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	practitioner_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	qualification_code VARCHAR,
	qualification_concept_id BIGINT,
	qualification_raw_code VARCHAR,
	qualification_raw_concept_id BIGINT,
	qualification_identifier_type VARCHAR,
	qualification_identifier_value VARCHAR,
	qualification_identifier_system VARCHAR,
	issuer_organisation_id BIGINT,
	issuer_name VARCHAR,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_practitioner_qualification ADD CONSTRAINT pk_practitioner_satpractqual_practqualid PRIMARY KEY (practitioner_qualification_id);
ALTER TABLE sat_practitioner_qualification ADD CONSTRAINT fk_practitioner_satpractqual_hubpractitioner_practitionerid FOREIGN KEY (practitioner_id) REFERENCES hub_practitioner(practitioner_id);

COMMENT ON COLUMN sat_practitioner_qualification.practitioner_id IS 'Practitioner Identity';
COMMENT ON COLUMN sat_practitioner_qualification.practitioner_qualification_id IS 'Practitioner Qualification Identity';
COMMENT ON COLUMN sat_practitioner_qualification.qualification_code IS 'Coded representation of the qualification. E.g. medical degree (MBBS, MBChB, MD), nursing degree (BSc Nursing), pharmacy degree (MPharm), dental degree (BDS). See https://build.fhir.org/practitioner-definitions.html#Practitioner.qualification.code';
COMMENT ON COLUMN sat_practitioner_qualification.qualification_identifier_type IS 'Type of the qualification identifier (e.g. certificate number, registration number)';
COMMENT ON COLUMN sat_practitioner_qualification.qualification_identifier_value IS 'The qualification identifier value. See https://build.fhir.org/practitioner-definitions.html#Practitioner.qualification.identifier';
COMMENT ON COLUMN sat_practitioner_qualification.qualification_identifier_system IS 'The namespace for the qualification identifier';
COMMENT ON COLUMN sat_practitioner_qualification.issuer_organisation_id IS 'Organization that regulates and issues the qualification (e.g. GMC, NMC, GPhC, HCPC, university). See https://build.fhir.org/practitioner-definitions.html#Practitioner.qualification.issuer';
COMMENT ON COLUMN sat_practitioner_qualification.issuer_name IS 'Name of the issuing organization (as free text)';
COMMENT ON COLUMN sat_practitioner_qualification.rank IS 'The recommended order the Practitioner Qualification entries should be presented';



-- -----------------------------------------------------------
-- create sat_practitioner_communication table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_practitioner_communication;

CREATE TABLE sat_practitioner_communication (
	practitioner_communication_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	practitioner_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	language_code VARCHAR,
	language_concept_id BIGINT,
	language_raw_code VARCHAR,
	language_raw_concept_id BIGINT,
	preferred BOOLEAN NOT NULL DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_practitioner_communication ADD CONSTRAINT pk_practitioner_satpractcomm_practcommid PRIMARY KEY (practitioner_communication_id);
ALTER TABLE sat_practitioner_communication ADD CONSTRAINT fk_practitioner_satpractcomm_hubpractitioner_practitionerid FOREIGN KEY (practitioner_id) REFERENCES hub_practitioner(practitioner_id);

COMMENT ON COLUMN sat_practitioner_communication.practitioner_id IS 'Practitioner Identity';
COMMENT ON COLUMN sat_practitioner_communication.practitioner_communication_id IS 'Practitioner Communication Identity';
COMMENT ON COLUMN sat_practitioner_communication.language_code IS 'A language which may be used to communicate with the practitioner. BCP-47 language tags (e.g. en, en-GB, cy, fr, ur, bn). See https://build.fhir.org/practitioner-definitions.html#Practitioner.communication.language';
COMMENT ON COLUMN sat_practitioner_communication.preferred IS 'Indicates whether this language is the practitioner''s preferred language for communication. See https://build.fhir.org/practitioner-definitions.html#Practitioner.communication.preferred';
COMMENT ON COLUMN sat_practitioner_communication.rank IS 'The recommended order the Practitioner Communication entries should be presented';



-- -----------------------------------------------------------
-- create sat_practitioner_role table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_practitioner_role;

CREATE TABLE sat_practitioner_role (
	practitioner_role_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	practitioner_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	active BOOLEAN DEFAULT TRUE,
	organisation_id BIGINT,
	organisation_ods_code VARCHAR,
	role_code VARCHAR,
	role_concept_id BIGINT,
	role_raw_code VARCHAR,
	role_raw_concept_id BIGINT,
	specialty_code VARCHAR,
	specialty_concept_id BIGINT,
	specialty_raw_code VARCHAR,
	specialty_raw_concept_id BIGINT,
	location_id BIGINT,
	healthcare_service_code VARCHAR,
	healthcare_service_concept_id BIGINT,
	healthcare_service_raw_code VARCHAR,
	healthcare_service_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_practitioner_role ADD CONSTRAINT pk_practitioner_satpractrole_practroleid PRIMARY KEY (practitioner_role_id);
ALTER TABLE sat_practitioner_role ADD CONSTRAINT fk_practitioner_satpractrole_hubpractitioner_practitionerid FOREIGN KEY (practitioner_id) REFERENCES hub_practitioner(practitioner_id);

COMMENT ON COLUMN sat_practitioner_role.practitioner_id IS 'Practitioner Identity';
COMMENT ON COLUMN sat_practitioner_role.practitioner_role_id IS 'Practitioner Role Identity';
COMMENT ON COLUMN sat_practitioner_role.active IS 'Whether this practitioner role record is in active use. See https://build.fhir.org/practitionerrole-definitions.html#PractitionerRole.active';
COMMENT ON COLUMN sat_practitioner_role.organisation_id IS 'Organization where the roles are available. Cross-schema reference to Organisation. See https://build.fhir.org/practitionerrole-definitions.html#PractitionerRole.organization';
COMMENT ON COLUMN sat_practitioner_role.organisation_ods_code IS 'The ODS (Organisation Data Service) code for the organisation. See https://digital.nhs.uk/services/organisation-data-service';
COMMENT ON COLUMN sat_practitioner_role.role_code IS 'Roles which this practitioner may perform. E.g. doctor, nurse, pharmacist, researcher, teacher, ict-professional. See https://hl7.org/fhir/R5/valueset-practitioner-role.html';
COMMENT ON COLUMN sat_practitioner_role.specialty_code IS 'Specific specialty of the practitioner. E.g. cardiology, dermatology, gastroenterology, general-practice, neurology, paediatrics, psychiatry, radiology, surgery. See https://build.fhir.org/practitionerrole-definitions.html#PractitionerRole.specialty';
COMMENT ON COLUMN sat_practitioner_role.location_id IS 'The location(s) at which this practitioner provides care. Cross-schema reference to Location. See https://build.fhir.org/practitionerrole-definitions.html#PractitionerRole.location';
COMMENT ON COLUMN sat_practitioner_role.healthcare_service_code IS 'The healthcare services provided through this role at the organization/location. See https://build.fhir.org/practitionerrole-definitions.html#PractitionerRole.healthcareService';
COMMENT ON COLUMN sat_practitioner_role.rank IS 'The recommended order the Practitioner Role entries should be presented';


-- -----------------------------------------------------------
-- create view_practitioners view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_practitioners AS
SELECT
    h.tennant_id,
    t.tennant_name,
    h.practitioner_id,
	h.practitioner_entity_id,
	h.person_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
    s.sat_practitioner_id,
    s.person_id,
    s.active,
    s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_practitioners.hub_practitioner h
    JOIN budd.qdp_practitioners.sat_practitioner s
      ON h.practitioner_id = s.practitioner_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_practitioners;


/**
-- -----------------------------------------------------------
-- create view_practitioners view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_practitioners AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.practitioner_id,
	h.practitioner_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_practitioner_id,
  s.individual_id,
  s.active,
  pn.given1 AS given_name,
  pn.family AS family_name,
  pn.prefix AS name_prefix,
  pn.suffix AS name_suffix,
  pn.full_name,
  s.birth_date,
  s.birth_datetime,
  s.birth_date_numeric,
  s.is_deceased,
  s.deceased_datetime,
  s.deceased_date_numeric,
  s.gender_code,
  s.gender_concept_id,
  s.gender_raw_code,
  s.gender_raw_concept_id,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_practitioners.hub_practitioner h
    JOIN budd.qdp_practitioners.sat_practitioner s
      ON h.practitioner_id = s.practitioner_id
    LEFT JOIN budd.qdp_practitioners.sat_practitioner_name pn
      ON h.practitioner_id = pn.practitioner_id AND pn.is_trusted = TRUE
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_practitioners;


-- -----------------------------------------------------------
-- create view_practitioner_names view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_practitioner_names AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.practitioner_id,
	h.practitioner_entity_id,
  s.gender_code,
  s.birth_date,
  s.active,
	pn.practitioner_name_id,
	pn.rank AS name_rank,
	pn.is_trusted AS is_trusted_name,
	pn.is_preferred AS is_preferred_name,
	pn.use_code AS name_use_code,
	pn.use_concept_id AS name_use_concept_id,
	pn.use_raw_code AS name_use_raw_code,
	pn.use_raw_concept_id AS name_use_raw_concept_id,
	pn.text AS name_text,
	pn.prefix AS name_prefix,
	pn.given1,
	pn.given1_soundex,
	pn.given2,
	pn.given2_soundex,
	pn.given3,
	pn.given3_soundex,
	pn.family,
	pn.family_soundex,
	pn.suffix AS name_suffix,
	pn.full_name,
	pn.full_name_soundex,
	pn.full_name_raw,
	pn.full_name_raw_soundex,
	pn.period_start AS name_period_start,
	pn.period_end AS name_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_practitioners.hub_practitioner h
    JOIN budd.qdp_practitioners.sat_practitioner s
      ON h.practitioner_id = s.practitioner_id
    JOIN budd.qdp_practitioners.sat_practitioner_name pn
      ON h.practitioner_id = pn.practitioner_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pn.rank
;

SELECT * from view_practitioner_names;


-- -----------------------------------------------------------
-- create view_practitioner_qualifications view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_practitioner_qualifications AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.practitioner_id,
	h.practitioner_entity_id,
  s.active,
  pn.given1 AS given_name,
  pn.family AS family_name,
  pn.prefix AS name_prefix,
	pq.practitioner_qualification_id,
	pq.qualification_code,
	pq.qualification_concept_id,
	pq.qualification_raw_code,
	pq.qualification_raw_concept_id,
	pq.qualification_identifier_type,
	pq.qualification_identifier_value,
	pq.qualification_identifier_system,
	pq.issuer_organisation_id,
	pq.issuer_name,
	pq.rank AS qualification_rank,
	pq.period_start AS qualification_period_start,
	pq.period_end AS qualification_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_practitioners.hub_practitioner h
    JOIN budd.qdp_practitioners.sat_practitioner s
      ON h.practitioner_id = s.practitioner_id
    LEFT JOIN budd.qdp_practitioners.sat_practitioner_name pn
      ON h.practitioner_id = pn.practitioner_id AND pn.is_trusted = TRUE
    JOIN budd.qdp_practitioners.sat_practitioner_qualification pq
      ON h.practitioner_id = pq.practitioner_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pq.rank
;

SELECT * from view_practitioner_qualifications;


-- -----------------------------------------------------------
-- create view_practitioner_roles view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_practitioner_roles AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.practitioner_id,
	h.practitioner_entity_id,
  s.active AS practitioner_active,
  pn.given1 AS given_name,
  pn.family AS family_name,
  pn.prefix AS name_prefix,
	pr.practitioner_role_id,
	pr.active AS role_active,
	pr.organisation_id,
	pr.organisation_ods_code,
	pr.role_code,
	pr.role_concept_id,
	pr.role_raw_code,
	pr.role_raw_concept_id,
	pr.specialty_code,
	pr.specialty_concept_id,
	pr.specialty_raw_code,
	pr.specialty_raw_concept_id,
	pr.location_id,
	pr.healthcare_service_code,
	pr.healthcare_service_concept_id,
	pr.healthcare_service_raw_code,
	pr.healthcare_service_raw_concept_id,
	pr.rank AS role_rank,
	pr.period_start AS role_period_start,
	pr.period_end AS role_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_practitioners.hub_practitioner h
    JOIN budd.qdp_practitioners.sat_practitioner s
      ON h.practitioner_id = s.practitioner_id
    LEFT JOIN budd.qdp_practitioners.sat_practitioner_name pn
      ON h.practitioner_id = pn.practitioner_id AND pn.is_trusted = TRUE
    JOIN budd.qdp_practitioners.sat_practitioner_role pr
      ON h.practitioner_id = pr.practitioner_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pr.rank
;

SELECT * from view_practitioner_roles;


**/
